# 🚀 Module 53 — Model Serving with FastAPI & Streamlit

## 1. Inference Pipeline

Now we're moving from **training ML/NLP models to deploying them for real-world usage**. This is an important step toward becoming an AI/ML engineer because a trained model sitting inside a notebook isn't yet a usable application.

## 1️⃣ What is model inference?

During training:

```text
Training Data
     ↓
Model
     ↓
Loss Calculation
     ↓
Backpropagation
     ↓
Weight Updates
```

During **inference**, the model receives new, unseen input and produces a prediction.

```text
New Input
    ↓
Trained Model
    ↓
Prediction
```

Inference = Using a trained model to generate predictions without updating its weights.

Examples:

- Sentiment analysis → Predict positive / negative.
- Image classification → Identify an object.
- Speech emotion recognition → Predict emotion.
- LLM → Generate a response.
- Fraud detection → Predict whether a transaction is suspicious.

## 2️⃣ The complete inference pipeline

**1. User Input**

Text, image, audio, JSON, etc.

**2. Validation**

Check input type, required fields and constraints.

**3. Preprocessing**

Transform input into model-compatible format.

**4. Model Inference**

Generate prediction using trained model.

**5. Postprocessing**

Convert raw output into understandable results.

**6. Response**

Return prediction through API or UI.

## 2️⃣ Step 1 — User Input

The user sends data to our application.

Examples:

| Application | User input |
|---|---|
| Sentiment analysis | `"This movie was amazing!"` |
| Image classifier | Image file |
| Chatbot | User prompt |
| Fraud detection | Transaction features |
| Speech emotion recognition | Audio file |

For our NLP model, the input could be:

```python
text = "This movie was amazing!"
```

But we cannot directly pass arbitrary raw user data into every model. It must follow the model's expected input format.

## 3️⃣ Step 2 — Validation

Validation ensures that the input is acceptable before the model processes it.

For example:

- Input is not empty.
- Input is the correct data type.
- Text isn't excessively long.
- Required fields exist.
- Uploaded files are valid.
- Input follows the expected schema.

```python
def validate_text(text):
    if not isinstance(text, str):
        raise TypeError("Input must be a string")

    if not text.strip():
        raise ValueError("Text cannot be empty")

    return text
```

Why validation matters:

Without validation, invalid inputs may cause application crashes, unnecessary model computation, or unexpected behavior.

## 4️⃣ Step 3 — Preprocessing

The model expects inputs in the same representation it was trained on.

For an NLP model:

```text
Raw Text
   ↓
Tokenizer
   ↓
Token IDs
   ↓
Attention Mask
   ↓
Tensor
   ↓
Model
```

Example:

```python
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

text = "This movie was amazing!"

inputs = tokenizer(
    text,
    truncation=True,
    padding=True,
    return_tensors="pt"
)

print(inputs)
```

The output generally contains:

- `input_ids`
- `attention_mask`

For an image model, preprocessing might instead involve resizing, normalization and converting the image into tensors.

Important: Preprocessing at inference time must remain compatible with the preprocessing used during training.

## 5️⃣ Step 4 — Model Inference

Now the preprocessed input enters the trained model.

Inference should generally be performed without gradient calculation.

### PyTorch

```python
import torch

model.eval()

with torch.inference_mode():
    outputs = model(**inputs)
```

### Why `model.eval()`?

It switches the model to evaluation mode, affecting modules such as:

- Dropout
- Batch Normalization

### Why `torch.inference_mode()`?

It disables autograd-related tracking during inference, reducing unnecessary computation and memory usage.

**Training:** Weight updates and gradient calculations.

**Inference:** Forward pass to produce predictions, without weight updates.

## 6️⃣ Step 5 — Postprocessing

Model outputs aren't always directly understandable by users.

For example, a classifier may output logits:

```python
tensor([[-1.2, 2.8]])
```

We convert these logits into probabilities.

```python
import torch

probabilities = torch.softmax(outputs.logits, dim=-1)

prediction = torch.argmax(probabilities, dim=-1)

confidence = probabilities.max().item()
```

Suppose:

```text
Logits:
[-1.2, 2.8]

Softmax:
[0.018, 0.982]

Prediction:
Positive

Confidence:
98.2%
```

We can convert the result into human-readable information:

```python
labels = ["Negative", "Positive"]

result = {
    "prediction": labels[prediction.item()],
    "confidence": round(confidence, 4)
}

print(result)
```

Output:

```python
{
    "prediction": "Positive",
    "confidence": 0.982
}
```

## 7️⃣ Step 6 — Response

Finally, we return the result to the user or another application.

The response could be:

- JSON from an API
- A web dashboard
- A mobile application response
- A chatbot message
- A displayed prediction with confidence

```json
{
    "status": "success",
    "prediction": "Positive",
    "confidence": 0.982
}
```

## 8️⃣ Where FastAPI and Streamlit fit

The inference pipeline is the **core logic**. FastAPI and Streamlit are tools used to expose that logic to users.

**User / Client**

Browser · Mobile App · Another Service

**FastAPI — Backend API**

Requests · Validation · API endpoints

**Inference Pipeline**

Preprocessing → Model → Postprocessing

**Prediction Response**

JSON output returned to the client

### FastAPI vs Streamlit

| Feature | FastAPI | Streamlit |
|---|---|---|
| Purpose | Backend API development | Interactive UI development |
| Interface | API endpoints | Web application |
| Input | HTTP requests / JSON | Text boxes, buttons, uploads |
| Output | JSON responses | UI elements, charts, results |
| Typical use | Production backend, model serving | ML demos, prototypes, dashboards |
| Documentation | Swagger UI | Interactive app interface |

### Typical industry architecture

```text
Frontend / Client
       │
       ▼
    FastAPI
       │
       ▼
  Input Validation
       │
       ▼
 Preprocessing
       │
       ▼
  ML Model (PyTorch)
       │
       ▼
 Postprocessing
       │
       ▼
    JSON Response
```

For a portfolio project, you can build a **Streamlit frontend → FastAPI backend → PyTorch model** architecture. This separates the interface from model-serving logic and makes the project more realistic.

## 🧠 What you should remember

- **Inference** is the process of using a trained model to make predictions on new data.
- The inference pipeline makes predictions reliable and consistent.
- Validation prevents invalid inputs from reaching the model.
- Preprocessing must be consistent with the model's training pipeline.
- `model.eval()` and `torch.inference_mode()` are important for efficient PyTorch inference.
- Postprocessing converts model outputs into understandable results.
- **FastAPI serves models through APIs; Streamlit builds interactive ML interfaces.**

---
